## LG Aimers 9기 — 전체 학습 및 제출 ZIP 생성 (Colab)

이 노트북은 검증에서 채택한 **02 LightGBM 65% + 03 CatBoost 35% logit 평균**을 2019~2024 전체 학습 데이터로 다시 학습하고, DACON 코드 제출용 ZIP을 만든다. 제출 모델을 고르는 검증은 이미 02·03의 forward OOF에서 끝냈으므로 이 단계에서 test 분포나 정답을 이용한 추가 보정은 하지 않는다.

LightGBM은 `game_type=R` 학습·half-life 2.5·선수 ID 제외 설정을 유지하고, CatBoost는 전체 `R/F` 학습·half-life 0.5·`season/game_type` 제외·선수 ID 포함 설정을 유지한다. 최종 파일은 `model/`, `script.py`, `requirements.txt`만 포함하며, 추론 피처는 각 테스트 행만으로 계산한다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path("/content/Aimers_9th"),
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/Aimers_9th"),
]
required_paths = [
    Path("data/train.csv"),
    Path("data/test.csv"),
    Path("data/sample_submission.csv"),
    Path("experiments/04_train_submit.py"),
    Path("src/features.py"),
    Path("submission/script.py"),
    Path("submission/requirements.txt"),
]
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidates
     if all((candidate / relative).exists() for relative in required_paths)),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트를 찾지 못했습니다. 04 노트북과 함께 experiments/04, "
        "submission 폴더, 최신 src/features.py를 Drive에 올렸는지 확인하세요."
    )
print("PROJECT_ROOT =", PROJECT_ROOT)

### 1. 제출 런타임 패키지 설치 및 GPU 확인

평가 서버 기본 패키지는 다시 고정하지 않고, 제출 스크립트에 필요한 LightGBM과 CatBoost만 설치한다. A100은 CatBoost **학습**에 사용하고, 만들어진 CatBoost native 모델은 평가 서버의 L4/CPU 추론에서 그대로 로드한다.

In [ ]:
requirements_path = PROJECT_ROOT / "submission/requirements.txt"
install = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--disable-pip-version-check", "-q", "-r", str(requirements_path)],
    capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError("LightGBM/CatBoost 설치에 실패했습니다.")

import catboost
import joblib
import lightgbm
import numpy as np
import pandas as pd

gpu_binary = shutil.which("nvidia-smi")
gpu_available = False
if gpu_binary:
    gpu_check = subprocess.run([gpu_binary, "-L"], capture_output=True, text=True)
    gpu_available = gpu_check.returncode == 0
    if gpu_available:
        print(gpu_check.stdout.strip())
print({
    "python": sys.version.split()[0],
    "lightgbm": lightgbm.__version__,
    "catboost": catboost.__version__,
    "pandas": pd.__version__,
    "gpu": gpu_available,
    "cpu_threads": os.cpu_count(),
})

### 2. 최종 학습 설정 확인

LightGBM 500회는 02의 고정 기준 설정이다. CatBoost 85회는 2025와 가장 가까운 2024 forward fold의 best iteration이며, 전체 학습에는 validation이 없으므로 조기 종료 대신 이 값을 고정한다. 혼합비 65:35는 2023·2024 pooled OOF와 시즌 교차 적용에서 모두 기존 02 LightGBM을 개선한 고정값이다.

In [ ]:
THREADS = max(1, min(8, os.cpu_count() or 2))
CATBOOST_TASK_TYPE = "GPU" if gpu_available else "CPU"
LGBM_ESTIMATORS = 500
CATBOOST_ITERATIONS = 85
OUTPUT_DIR = PROJECT_ROOT / "outputs/final_submission_v1"

if CATBOOST_TASK_TYPE == "CPU":
    print("경고: GPU가 없어 CatBoost를 CPU로 학습합니다. 결과 형식은 같지만 더 오래 걸릴 수 있습니다.")
print({
    "threads": THREADS,
    "catboost_task_type": CATBOOST_TASK_TYPE,
    "lgbm_estimators": LGBM_ESTIMATORS,
    "catboost_iterations": CATBOOST_ITERATIONS,
    "blend": "logit(LightGBM 0.65 + CatBoost 0.35)",
    "output_dir": str(OUTPUT_DIR),
})

### 3. 입력 데이터 사전 점검

학습 타깃·시즌·경기 유형과 test/sample의 `row_id` 대응을 학습 전에 확인한다. 여기서는 test의 통계로 모델이나 확률을 보정하지 않는다.

In [ ]:
train_check = pd.read_csv(
    PROJECT_ROOT / "data/train.csv",
    usecols=["row_id", "season", "game_type", "control_success"],
    encoding="utf-8-sig",
)
test_ids = pd.read_csv(
    PROJECT_ROOT / "data/test.csv", usecols=["row_id"], encoding="utf-8-sig"
)
sample = pd.read_csv(PROJECT_ROOT / "data/sample_submission.csv", encoding="utf-8-sig")
assert set(train_check["control_success"].unique()) <= {0, 1}
assert train_check["row_id"].is_unique
assert test_ids["row_id"].is_unique and sample["row_id"].is_unique
assert set(test_ids["row_id"]) == set(sample["row_id"])
assert list(sample.columns) == ["row_id", "control_success"]
display(
    train_check.groupby(["season", "game_type"], observed=True)
    .agg(rows=("row_id", "size"), target_rate=("control_success", "mean"))
    .reset_index()
)
print(f"train={len(train_check):,} test={len(test_ids):,} sample={len(sample):,}")
del train_check, test_ids, sample

### 4. 전체 데이터 학습 및 제출 패키지 생성

이 셀은 두 모델 학습, native 모델 저장, 65:35 logit 앙상블 설정 저장, 평가용 `script.py` 실행, ZIP 생성을 연속 수행한다. 학습 후에는 저장 전후 예측과 행 순서 변경·분할·단일 행 예측의 일치 여부도 검사한다.

In [ ]:
cmd = [
    sys.executable, str(PROJECT_ROOT / "experiments/04_train_submit.py"),
    "--train-path", str(PROJECT_ROOT / "data/train.csv"),
    "--data-dir", str(PROJECT_ROOT / "data"),
    "--output-dir", str(OUTPUT_DIR),
    "--threads", str(THREADS),
    "--lgbm-estimators", str(LGBM_ESTIMATORS),
    "--catboost-iterations", str(CATBOOST_ITERATIONS),
    "--catboost-task-type", CATBOOST_TASK_TYPE,
]
print("실행:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 5. 학습 결과와 자동 검증 확인

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / "training_summary.json"
with SUMMARY_PATH.open(encoding="utf-8") as handle:
    summary = json.load(handle)

model_rows = []
for name in ["lgbm", "catboost"]:
    values = summary[name]
    model_rows.append({
        "model": name,
        "train_rows": values["train_rows"],
        "train_game_type": values["train_game_type"],
        "half_life": values["half_life"],
        "features": values["feature_count"],
        "categoricals": values["categorical_count"],
        "train_seconds": values["train_seconds"],
    })
display(pd.DataFrame(model_rows))
display(pd.Series(summary["audit"], name="value").to_frame())
display(pd.Series(summary["end_to_end"], name="value").to_frame())
assert max(
    summary["audit"]["shuffle_max_abs_diff"],
    summary["audit"]["chunk_max_abs_diff"],
    summary["audit"]["single_row_max_abs_diff"],
) <= 1e-12
assert summary["end_to_end"]["returncode"] == 0
print("자동 검증: 통과")

### 6. 제출 ZIP 구조·크기·해시 확인

ZIP의 최상위에는 `model/`, `script.py`, `requirements.txt`만 있어야 한다. `data/`, `output/`, 노트북, 소스 폴더, `__pycache__`는 넣지 않는다.

In [ ]:
SUBMIT_ZIP = Path(summary["package"]["zip_path"])
EXPECTED = {
    "model/lgbm_model.txt",
    "model/catboost_model.cbm",
    "model/artifacts.joblib",
    "script.py",
    "requirements.txt",
}
with zipfile.ZipFile(SUBMIT_ZIP) as archive:
    names = set(archive.namelist())
    members = pd.DataFrame([
        {"name": item.filename, "bytes": item.file_size, "compressed": item.compress_size}
        for item in archive.infolist()
    ])
    assert archive.testzip() is None
assert names == EXPECTED
assert SUBMIT_ZIP.stat().st_size < 10 * 1024**3
display(members)
print("제출 파일 =", SUBMIT_ZIP)
print("ZIP 크기 =", f"{SUBMIT_ZIP.stat().st_size / 1024**2:.2f} MiB")
print("SHA-256 =", summary["package"]["zip_sha256"])

### 7. 제출 및 다음 실험

DACON에는 위 셀의 `submit_01.zip` **파일 자체를 압축 해제하지 않고** 업로드한다. 파일명은 확장자를 포함해 13자로, 30자 제한을 충족한다. `training_summary.json`, 피처 중요도 CSV, 실행 완료 노트북은 검토용이며 제출하지 않는다.

이 제출은 현재 OOF 근거가 있는 앵커 점수다. Public 점수와 제출 성공 여부를 기록한 뒤, 같은 검증 체계를 유지하면서 CatBoost iteration·최근성 가중치·`F` 취약 구간을 한 변수씩 개선한다. Public 점수만 보고 혼합비를 반복 조정하지 않는다.